# Day 28 - Structured Streaming Basics

**Topic:** Structured Streaming Basics  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** เข้าใจพื้นฐาน `readStream`, `writeStream`, source, sink, trigger, output mode, micro-batch และ checkpoint ใน Structured Streaming

วันนี้เราจะเรียนให้เห็นภาพว่า Structured Streaming ไม่ใช่แค่การเขียน loop อ่านไฟล์เองเรื่อย ๆ แต่เป็นวิธีที่ Spark ให้เราเขียน logic คล้าย batch DataFrame แล้วให้ Spark execute แบบ incremental เมื่อมีข้อมูลใหม่เข้ามา โดยใน lab นี้จะใช้ตัวอย่างเล็ก ๆ ที่เหมาะกับ Microsoft Fabric Notebook ก่อน แล้วค่อยต่อยอดเป็น Lakehouse streaming pattern ใน Day 29

## Goal of the day

1. อธิบายได้ว่า **Structured Streaming** คืออะไรในมุมของ Spark
2. แยกให้ออกว่าอะไรคือ **streaming source**, **streaming DataFrame** และ **streaming sink**
3. เข้าใจ role ของ `readStream`, `writeStream`, `trigger`, `outputMode` และ `checkpointLocation`
4. ใช้ `.isStreaming` เพื่อตรวจสอบว่า DataFrame เป็น batch หรือ streaming ได้
5. เริ่มระวังว่า streaming query ต้องมี lifecycle เช่น `start()`, monitoring และ `stop()` ไม่ใช่ run แล้วจบทันทีเหมือน batch job

## Why it matters in production

ใน production pipeline ข้อมูลบางประเภทไม่ได้มาเป็น batch วันละครั้งเสมอไป เช่น event log, clickstream, payment event, IoT event หรือ near real-time transaction feed การเข้าใจ Structured Streaming ช่วยให้ Data Engineer อธิบายและออกแบบ pipeline ที่ process ข้อมูลใหม่แบบต่อเนื่องได้โดยไม่ต้องเขียน manual loop เอง

ประเด็นที่ต้องเข้าใจตั้งแต่พื้นฐาน:

- Streaming job เป็น long-running query หรือ query ที่รันซ้ำเป็น micro-batch ตาม trigger
- `readStream` สร้าง streaming DataFrame แต่ยังไม่เริ่มอ่านจริงจนกว่าจะมีการเรียก `writeStream.start()`
- output mode ต้องสอดคล้องกับ transformation เช่น append สำหรับ new rows, complete สำหรับ aggregation ที่รองรับ
- checkpoint สำคัญมากสำหรับ progress tracking และ fault-tolerant restart / recovery
- memory sink เหมาะกับ lab/debug เท่านั้น ไม่ควรใช้ใน production

Production takeaway:

> Structured Streaming ให้เราเขียน logic คล้าย batch แต่ Spark execute แบบ incremental ตามข้อมูลใหม่ โดยใช้ checkpoint เพื่อเก็บ progress และรองรับ fault tolerance


## Key concepts

| Concept | Meaning |
|---|---|
| Structured Streaming | Spark API สำหรับ process streaming data ด้วย DataFrame/Spark SQL style |
| Streaming DataFrame | DataFrame ที่ represent ข้อมูลที่ยังไหลเข้ามาเรื่อย ๆ ตรวจสอบได้ด้วย `.isStreaming` |
| Streaming source | แหล่งข้อมูลขาเข้า เช่น file source, rate source หรือ Kafka/Event Hub ขึ้นอยู่กับ connector ที่ runtime รองรับ |
| Streaming sink | ปลายทางของ streaming query เช่น memory sink, console sink, Delta/Lakehouse table |
| Micro-batch | Spark แบ่งข้อมูลใหม่ที่เข้ามาเป็น batch เล็ก ๆ แล้ว process ต่อเนื่อง |
| Trigger | กำหนดว่า Spark จะ check/process ข้อมูลใหม่บ่อยแค่ไหน |
| Output mode | วิธีเขียนผลลัพธ์ออก เช่น `append`, `update`, `complete` |
| Checkpoint | ตำแหน่งเก็บ progress/state ของ streaming query เพื่อรองรับ restart และ fault-tolerant recovery |
| Streaming query lifecycle | flow ตั้งแต่ build query → `start()` → monitor → `stop()` |
| Memory sink | sink สำหรับทดลอง/debug ใน notebook โดย Spark จะเก็บผลลัพธ์ไว้เป็น in-memory table ตาม `queryName` ทำให้ query ดูผลผ่าน `spark.sql()` ได้ แต่ไม่ใช่ production storage |


## Concept explanation

### Structured Streaming คืออะไร?

Structured Streaming คือ Spark engine สำหรับ streaming data ที่ให้เราเขียน logic ด้วย DataFrame API คล้าย batch processing เช่น `select`, `withColumn`, `filter`, `groupBy` แล้ว Spark จะ execute logic นี้กับข้อมูลที่เข้ามาใหม่แบบ incremental

พูดง่าย ๆ:

> Batch DataFrame = ข้อมูลมีขอบเขตชัด อ่านแล้ว process จบ  
> Streaming DataFrame = ข้อมูลยังมาเรื่อย ๆ ต้องมี query lifecycle และ checkpoint

ตัวอย่าง mindset:

```python
# Batch
spark.read.format("csv").load(path)

# Streaming
spark.readStream.format("...").load(path)
```

### `readStream` ยังไม่เริ่ม execute ทันที

เหมือน Day 02 เรื่อง Lazy Evaluation: การสร้าง streaming DataFrame ยังไม่ได้เริ่มอ่านข้อมูลจริงทันที Spark แค่รู้ว่า source อยู่ที่ไหนและ schema/logic เป็นอย่างไร

จุดที่เริ่ม streaming query จริงคือ:

```python
query = df_stream.writeStream.start()
```

### Source → Transform → Sink

Structured Streaming มองเป็น flow ง่าย ๆ:

1. อ่านจาก streaming source ด้วย `readStream`
2. transform ด้วย DataFrame API
3. เขียนออกด้วย `writeStream`
4. ใช้ checkpoint เพื่อติดตาม progress

### Trigger และ micro-batch

โดยทั่วไป Spark จะ process streaming data แบบ micro-batch คือรวม records ใหม่ที่เข้ามาในช่วงเวลาหนึ่ง แล้ว process เป็น batch เล็ก ๆ ตาม trigger ที่ตั้งไว้ เช่น ทุก 5 วินาที

ตัวอย่าง:

```python
.trigger(processingTime="5 seconds")
```

### Output mode ใช้ต่างกันตาม logic

| Output mode | ใช้เมื่อไหร่ |
|---|---|
| `append` | เขียนเฉพาะ rows ใหม่ที่ไม่ต้องอัปเดตย้อนหลังแล้ว เหมาะกับ append-only events หรือ window aggregation ที่ใช้ watermark |
| `update` | เขียนเฉพาะ rows ของผลลัพธ์ที่เปลี่ยนใน micro-batch นั้น เหมาะกับ aggregation ที่ผลลัพธ์เดิมยังเปลี่ยนแปลงได้ แต่บาง sink อาจไม่ support |
| `complete` | เขียน result table ทั้งหมดทุก micro-batch แม้บาง rows จะไม่ได้เปลี่ยน เหมาะกับ aggregation ขนาดเล็กใน lab/debug ไม่เหมาะกับ result ใหญ่ |

### Checkpoint คือหัวใจของ recovery

Checkpoint ใช้เก็บ progress ของ streaming query เช่น source อ่านถึง offset ไหนแล้ว และในบางกรณีจะเก็บ state ของ operation ที่มี state ด้วย เช่น aggregation หรือ watermark

ถ้า streaming query หยุดแล้วเริ่มใหม่ Spark สามารถใช้ checkpoint เดิมเพื่อ resume ต่อจากจุดล่าสุดได้ แทนที่จะเริ่มอ่านใหม่ตั้งแต่ต้น

ใน lab นี้เราอาจลบ checkpoint เพื่อ reset ตัวอย่างได้ แต่ใน production ไม่ควรลบ checkpoint โดยไม่เข้าใจผลกระทบ เพราะอาจทำให้ข้อมูลถูก reprocess, duplicate หรือทำให้ state ของ streaming query หายได้


## Mermaid diagram: Structured Streaming Micro-batch Flow

```mermaid
flowchart LR
    A[Streaming Source<br/>new events] --> B[readStream]
    B --> C[Streaming DataFrame]
    C --> D[Transform<br/>select / withColumn / filter / groupBy]
    D --> E[writeStream]
    E --> F[Streaming Sink<br/>memory / Delta / Lakehouse]
    E --> G[Checkpoint<br/>offset / progress + state]
    H[Trigger<br/>every N seconds] -. starts each micro-batch .-> B

    B -. micro-batch 1 .-> C
    B -. micro-batch 2 .-> C
    B -. micro-batch 3 .-> C
```

Key idea:

> เราเขียน transformation คล้าย batch แต่ Spark จะ process ข้อมูลใหม่เป็น micro-batches ตาม trigger และใช้ checkpoint เพื่อจำ progress ว่าอ่าน source ถึงไหนแล้ว รวมถึงช่วย resume/recover เมื่อ query เริ่มใหม่

## Setup / imports

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

import time

StatementMeta(, 795e056b-5a93-4d00-b892-9a3d6321b045, 3, Finished, Available, Finished, False)

## Create mock data

ก่อนเริ่ม streaming จริง เราจะสร้าง batch DataFrame เล็ก ๆ เพื่อเปรียบเทียบกับ streaming DataFrame เพราะ concept สำคัญของวันนี้คือ:

> Batch DataFrame และ Streaming DataFrame ใช้ transformation API คล้ายกัน แต่ execution lifecycle ไม่เหมือนกัน

In [2]:
static_events_data = [
    ("E001", "mobile_app", "click", "2026-06-01 09:00:00", 101),
    ("E002", "mobile_app", "view", "2026-06-01 09:01:00", 102),
    ("E003", "web", "purchase", "2026-06-01 09:02:00", 101),
    ("E004", "web", "click", "2026-06-01 09:03:00", 103),
]

static_events_schema = T.StructType([
    T.StructField("event_id", T.StringType(), False),
    T.StructField("source_system", T.StringType(), False),
    T.StructField("event_type", T.StringType(), False),
    T.StructField("event_timestamp", T.StringType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
])

df_static_events = spark.createDataFrame(static_events_data, static_events_schema)

df_static_events = df_static_events.withColumn(
    "event_timestamp",
    F.to_timestamp("event_timestamp")
)

df_static_events.show(truncate=False)
df_static_events.printSchema()
print("Is static DataFrame streaming?:", df_static_events.isStreaming)

StatementMeta(, 795e056b-5a93-4d00-b892-9a3d6321b045, 4, Finished, Available, Finished, False)

+--------+-------------+----------+-------------------+-----------+
|event_id|source_system|event_type|event_timestamp    |customer_id|
+--------+-------------+----------+-------------------+-----------+
|E001    |mobile_app   |click     |2026-06-01 09:00:00|101        |
|E002    |mobile_app   |view      |2026-06-01 09:01:00|102        |
|E003    |web          |purchase  |2026-06-01 09:02:00|101        |
|E004    |web          |click     |2026-06-01 09:03:00|103        |
+--------+-------------+----------+-------------------+-----------+

root
 |-- event_id: string (nullable = false)
 |-- source_system: string (nullable = false)
 |-- event_type: string (nullable = false)
 |-- event_timestamp: timestamp (nullable = true)
 |-- customer_id: integer (nullable = false)

Is static DataFrame streaming?: False


## PySpark code examples

ใน section นี้จะไล่จาก batch vs streaming DataFrame → `readStream` → transformation → `writeStream` → query monitoring → output mode เบื้องต้น

### Example 1: Batch DataFrame has bounded data

Batch DataFrame มีข้อมูลเป็นชุดที่ชัดเจน เมื่อเรียก action เช่น `.show()` หรือ `.count()` Spark execute แล้วจบ

Cell นี้ย้ำว่า `df_static_events.isStreaming` ต้องเป็น `False`

In [3]:
print("Batch DataFrame is streaming?:", df_static_events.isStreaming)
print("Batch row count:", df_static_events.count())

df_static_events.groupBy("source_system", "event_type").count().orderBy("source_system", "event_type").show()

StatementMeta(, 795e056b-5a93-4d00-b892-9a3d6321b045, 5, Finished, Available, Finished, False)

Batch DataFrame is streaming?: False
Batch row count: 4
+-------------+----------+-----+
|source_system|event_type|count|
+-------------+----------+-----+
|   mobile_app|     click|    1|
|   mobile_app|      view|    1|
|          web|     click|    1|
|          web|  purchase|    1|
+-------------+----------+-----+



### Example 2: Create a streaming DataFrame with rate source

`rate` source เป็น built-in streaming source ของ Spark สำหรับทดลอง Structured Streaming โดย Spark จะ generate columns หลัก ๆ:

- `timestamp`
- `value`

ใน production source จริงอาจเป็น file source, Kafka/Event Hub connector หรือระบบ event อื่น ๆ ขึ้นกับ runtime/connector ที่ใช้งาน

Cell นี้ยังไม่เริ่ม streaming query จริง แค่สร้าง streaming DataFrame และตรวจ `.isStreaming`

In [4]:
df_rate_raw = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 3)
    .load()
)

print("Rate DataFrame is streaming?:", df_rate_raw.isStreaming)
df_rate_raw.printSchema()

StatementMeta(, 795e056b-5a93-4d00-b892-9a3d6321b045, 6, Finished, Available, Finished, False)

Rate DataFrame is streaming?: True
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)



### Example 3: Transform streaming DataFrame

เราสามารถใช้ DataFrame transformation หลายตัวกับ streaming DataFrame ได้ เช่น `select`, `withColumn`, `filter`

ข้อสำคัญคือ transformation เหล่านี้ยังไม่ execute จนกว่าจะมี `writeStream.start()`

In [5]:
df_rate_events = (
    df_rate_raw
    .select(
        F.col("value").cast("long").alias("event_sequence"),
        F.col("timestamp").alias("event_timestamp"),
    )
    .withColumn("event_id", F.concat(F.lit("EVT-"), F.lpad(F.col("event_sequence"), 6, "0")))
    .withColumn("source_system", F.lit("rate_source"))
    .withColumn(
        "event_type",
        F.when((F.col("event_sequence") % 3) == 0, F.lit("click"))
         .when((F.col("event_sequence") % 3) == 1, F.lit("view"))
         .otherwise(F.lit("purchase"))
    )
    .select("event_id", "source_system", "event_type", "event_timestamp", "event_sequence")
)

# Tips:
# - lpad(col, len, pad): makes a string reach the target length by adding padding characters on the left
#   Example: lpad("123", 5, "0") -> "00123"
#   Useful for standardizing IDs or codes to the same fixed length

print("Transformed DataFrame is streaming?:", df_rate_events.isStreaming)
df_rate_events.printSchema()

StatementMeta(, 795e056b-5a93-4d00-b892-9a3d6321b045, 7, Finished, Available, Finished, False)

Transformed DataFrame is streaming?: True
root
 |-- event_id: string (nullable = true)
 |-- source_system: string (nullable = false)
 |-- event_type: string (nullable = false)
 |-- event_timestamp: timestamp (nullable = true)
 |-- event_sequence: long (nullable = true)



### Example 4: Start a streaming query to memory sink

ตัวอย่างนี้ใช้ `memory` sink เพื่อให้ notebook query ผลลัพธ์ด้วย Spark SQL ได้ง่าย

ข้อควรจำ:

- `memory` sink เหมาะกับ lab/debug เท่านั้น
- `query.start()` คือจุดที่ streaming query เริ่มทำงานจริง
- ต้อง stop query เมื่อทดลองจบ เพื่อไม่ให้มี active streaming query ค้างอยู่ใน session

In [ ]:
stream_query_name = "day28_rate_events_memory"
checkpoint_path = "Files/day28_structured_streaming_basics/checkpoints/rate_events_memory"

# Stop previous query with the same name if it is still active.
# spark.streams.active contains active StreamingQuery objects in the current Spark session.
for active_query in spark.streams.active:
    if active_query.name == stream_query_name:
        active_query.stop()

# For lab reset only. Do not delete checkpoint blindly in production.
try:
    mssparkutils.fs.rm(checkpoint_path, True)
except Exception as e:
    print("Checkpoint cleanup skipped. Expected on first run.")

query_events = (
    df_rate_events
    .writeStream
    .format("memory")
    .queryName(stream_query_name)
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(processingTime="5 seconds")
    .start()
)

# Pauses Python for 12 seconds while the streaming query continues running in the background,
# allowing Spark to complete at least 2 micro-batches before we stop the query.
time.sleep(12)

print("Query name:", query_events.name)
print("Query is active before stop?:", query_events.isActive)
print("Query status:", query_events.status)

query_events.stop()
print("Query is active after stop?:", query_events.isActive)

spark.sql(f"SELECT * FROM {stream_query_name} ORDER BY event_sequence LIMIT 20").show(truncate=False)

StatementMeta(, 795e056b-5a93-4d00-b892-9a3d6321b045, 9, Finished, Available, Finished, False)

Query name: day28_rate_events_memory
Query is active before stop?: True
Query status: {'message': 'Waiting for next trigger', 'isDataAvailable': True, 'isTriggerActive': False}
Query is active after stop?: False
+----------+-------------+----------+-----------------------+--------------+
|event_id  |source_system|event_type|event_timestamp        |event_sequence|
+----------+-------------+----------+-----------------------+--------------+
|EVT-000000|rate_source  |click     |2026-06-14 06:43:20.516|0             |
|EVT-000001|rate_source  |view      |2026-06-14 06:43:20.849|1             |
|EVT-000002|rate_source  |purchase  |2026-06-14 06:43:21.183|2             |
|EVT-000003|rate_source  |click     |2026-06-14 06:43:21.516|3             |
|EVT-000004|rate_source  |view      |2026-06-14 06:43:21.849|4             |
|EVT-000005|rate_source  |purchase  |2026-06-14 06:43:22.183|5             |
|EVT-000006|rate_source  |click     |2026-06-14 06:43:22.516|6             |
|EVT-000007|rate_s

> หมายเหตุ: `query.status` เป็น snapshot ณ ตอนที่ print ออกมา ถ้าเห็น `Waiting for next trigger` และ `isTriggerActive: False` แปลว่า micro-batch รอบล่าสุดประมวลผลเสร็จแล้ว และ Spark กำลังรอประมวลผลรอบถัดไปตาม trigger interval

### Example 5: Inspect recent progress

Streaming query มี runtime information เช่น batch id, input rows, processed rows per second และ duration ของ micro-batch

ใน production เราไม่ควรดูแค่ output table แต่ต้องใช้ข้อมูลเหล่านี้ช่วย monitor ว่า streaming query ยังรันปกติไหม ประมวลผลทันไหม และมี failure เกิดขึ้นหรือเปล่า

In [9]:
# recentProgress is a list of recent streaming micro-batch progress records.
# [-3:] is Python list slicing that keeps only the latest 3 items from the list.
# Example: ["batch 0", "batch 1", "batch 2", "batch 3"][-3:] returns ["batch 1", "batch 2", "batch 3"].
for progress in query_events.recentProgress[-3:]:
    print("batchId:", progress.get("batchId"))
    print("numInputRows:", progress.get("numInputRows"))
    print("processedRowsPerSecond:", progress.get("processedRowsPerSecond"))
    print("durationMs:", progress.get("durationMs"))
    print("---")

StatementMeta(, 795e056b-5a93-4d00-b892-9a3d6321b045, 11, Finished, Available, Finished, False)

batchId: 0
numInputRows: 0
processedRowsPerSecond: 0.0
durationMs: {'addBatch': 60, 'commitOffsets': 377, 'getBatch': 0, 'latestOffset': 0, 'queryPlanning': 15, 'triggerExecution': 788, 'walCommit': 301}
---
batchId: 1
numInputRows: 12
processedRowsPerSecond: 16.71309192200557
durationMs: {'addBatch': 93, 'commitOffsets': 347, 'getBatch': 0, 'latestOffset': 0, 'queryPlanning': 14, 'triggerExecution': 718, 'walCommit': 262}
---
batchId: 2
numInputRows: 15
processedRowsPerSecond: 23.006134969325153
durationMs: {'addBatch': 81, 'commitOffsets': 271, 'getBatch': 0, 'latestOffset': 0, 'queryPlanning': 11, 'triggerExecution': 652, 'walCommit': 289}
---


> หมายเหตุ: batchId 0 มี numInputRows = 0 เสมอ เพราะเป็น initialization batch ที่ Spark ใช้ setup query และ checkpoint ก่อนเริ่มอ่าน records จริงใน batchId 1 เป็นต้นไป

### Example 6: Complete output mode with aggregation

ถ้ามี streaming aggregation เช่น นับจำนวน events ต่อ `event_type` ผลลัพธ์ aggregate ของแต่ละ `event_type` อาจเปลี่ยนได้เมื่อมี event ใหม่เข้ามา จึงต้องเลือก output mode ให้เหมาะสม

ตัวอย่างนี้ใช้ `complete` mode เพื่อเขียน aggregate result ทั้งหมดเข้า memory sink ทุก micro-batch

In [10]:
df_event_type_counts = (
    df_rate_events
    .groupBy("event_type")
    .agg(F.count("*").alias("event_count"))
)

print("Aggregated streaming DataFrame is streaming?:", df_event_type_counts.isStreaming)
df_event_type_counts.printSchema()

StatementMeta(, 795e056b-5a93-4d00-b892-9a3d6321b045, 12, Finished, Available, Finished, False)

Aggregated streaming DataFrame is streaming?: True
root
 |-- event_type: string (nullable = false)
 |-- event_count: long (nullable = false)



In [ ]:
counts_query_name = "day28_event_type_counts_memory"
counts_checkpoint_path = "Files/day28_structured_streaming_basics/checkpoints/event_type_counts_memory"

for active_query in spark.streams.active:
    if active_query.name == counts_query_name:
        active_query.stop()

# For lab reset only. Do not delete checkpoint blindly in production.
try:
    mssparkutils.fs.rm(counts_checkpoint_path, True)
except Exception as e:
    print("Checkpoint cleanup skipped. Expected on first run.")

query_counts = (
    df_event_type_counts
    .writeStream
    .format("memory")
    .queryName(counts_query_name)
    .outputMode("complete")
    .option("checkpointLocation", counts_checkpoint_path)
    .trigger(processingTime="5 seconds")
    .start()
)

# Wait long enough for at least one micro-batch to finish and commit results to the memory sink.
# Note: with complete mode + aggregation, a short wait time may not be enough.
# Stopping the query before the first successful commit can result in an empty memory sink.
time.sleep(30)

print("Query name:", query_counts.name)
print("Query is active before stop?:", query_counts.isActive)
print("Query status:", query_counts.status)

query_counts.stop()
print("Query is active after stop?:", query_counts.isActive)

spark.sql(f"SELECT * FROM {counts_query_name} ORDER BY event_type").show(truncate=False)

StatementMeta(, 795e056b-5a93-4d00-b892-9a3d6321b045, 17, Finished, Available, Finished, False)

Query name: day28_event_type_counts_memory
Query is active before stop?: True
Query status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Query is active after stop?: False
+----------+-----------+
|event_type|event_count|
+----------+-----------+
|click     |12         |
|purchase  |12         |
|view      |12         |
+----------+-----------+



> หมายเหตุ: `query.status` เป็น snapshot ณ ตอนที่ print ถ้าเห็น `Processing new data` และ `isTriggerActive: True` แปลว่า query กำลัง process micro-batch อยู่พอดี ส่วนผลลัพธ์ที่ query จาก memory sink อาจเป็นผลจาก micro-batch ก่อนหน้าที่ commit สำเร็จแล้ว

### Example 7: Compare batch and streaming mental model

จุดที่คล้ายกัน:

- ใช้ DataFrame API เหมือนกันหลายตัว
- สร้าง transformation chain ได้
- ใช้ schema และ column expression ได้

จุดที่ต่างกัน:

- Batch ต้องมี action เช่น `.count()`, `.show()`, `.write`
- Streaming query ต้องมี sink และจะเริ่มทำงานจริงเมื่อเรียก `writeStream.start()`
- Streaming query ต้อง monitor และ stop/restart อย่างมีวินัย

In [17]:
comparison_rows = [
    ("Create input DataFrame", "spark.read", "spark.readStream"),
    ("Transform", "select/filter/withColumn/groupBy", "Similar transformations, but aggregation needs output mode/checkpoint/state handling"),
    ("Start execution", "Action: show/count/write", "writeStream.start()"),
    ("Output", "DataFrame result or table/file", "Streaming sink"),
    ("Progress tracking", "Job/stage/task metrics", "Streaming query progress + checkpoint"),
    ("Lifecycle", "Runs once and finishes", "Keeps running until stopped, failed, or session ends"),
]

comparison_schema = ["Area", "Batch DataFrame", "Streaming DataFrame"]

df_batch_vs_streaming = spark.createDataFrame(comparison_rows, comparison_schema)
df_batch_vs_streaming.show(truncate=False)

StatementMeta(, 795e056b-5a93-4d00-b892-9a3d6321b045, 19, Finished, Available, Finished, False)

+----------------------+--------------------------------+------------------------------------------------------------------------------------+
|Area                  |Batch DataFrame                 |Streaming DataFrame                                                                 |
+----------------------+--------------------------------+------------------------------------------------------------------------------------+
|Create input DataFrame|spark.read                      |spark.readStream                                                                    |
|Transform             |select/filter/withColumn/groupBy|Similar transformations, but aggregation needs output mode/checkpoint/state handling|
|Start execution       |Action: show/count/write        |writeStream.start()                                                                 |
|Output                |DataFrame result or table/file  |Streaming sink                                                                      |

## Quick recap

| Question | Answer |
|---|---|
| Structured Streaming คืออะไร? | การเขียน streaming pipeline ด้วย DataFrame/Spark SQL style แล้วให้ Spark process incremental data |
| `readStream` ทำอะไร? | สร้าง streaming DataFrame จาก streaming source แต่ยังไม่เริ่ม query จริง |
| เริ่ม streaming query ตอนไหน? | ตอนเรียก `writeStream.start()` |
| Micro-batch คืออะไร? | การแบ่งข้อมูลใหม่เป็น batch เล็ก ๆ เพื่อ process ต่อเนื่อง |
| `outputMode("append")` เหมาะกับอะไร? | append-only event rows หรือผลลัพธ์ที่ไม่ต้องอัปเดตย้อนหลังแล้ว เช่น window aggregation ที่ใช้ watermark |
| Checkpoint สำคัญยังไง? | เก็บ progress ว่าอ่าน source ถึงไหนแล้ว และเก็บ state ของ stateful operations เช่น aggregation/window เพื่อใช้ตอน restart หรือ recovery |

## Exercises

### Exercise 1: Create a streaming DataFrame and check schema

สร้าง streaming DataFrame ชื่อ `df_ex_rate_raw` จาก `rate` source

Requirements:

- ใช้ `rowsPerSecond = 2`
- ตรวจ `.isStreaming`
- แสดง schema ด้วย `.printSchema()`
- ยังไม่ต้อง start query

Expected result:

- `df_ex_rate_raw.isStreaming` เป็น `True`
- schema มี columns `timestamp` และ `value`

In [18]:
df_ex_rate_raw = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 2)
    .load()
)

print("Exercise 1 - Is streaming?:", df_ex_rate_raw.isStreaming)
df_ex_rate_raw.printSchema()

StatementMeta(, 795e056b-5a93-4d00-b892-9a3d6321b045, 20, Finished, Available, Finished, False)

Exercise 1 - Is streaming?: True
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)



### Exercise 2: Transform streaming events and write to memory sink

สร้าง streaming DataFrame ชื่อ `df_ex_events` จาก `df_ex_rate_raw` แล้วเขียนไป memory sink

Requirements:

- สร้าง `event_id` จาก `value`
- สร้าง `event_timestamp` จาก `timestamp`
- สร้าง `event_type` โดยให้เลขคู่เป็น `view` และเลขคี่เป็น `click`
- ใช้ `outputMode("append")`
- ตั้ง `checkpointLocation`
- run สั้น ๆ แล้ว `stop()` query

Expected result:

- query start ได้
- query ถูก stop หลังทดลอง
- query memory table แล้วเห็น event rows

In [19]:
df_ex_events = (
    df_ex_rate_raw
    .select(
        F.col("value").cast("long").alias("event_sequence"),
        F.col("timestamp").alias("event_timestamp"),
    )
    .withColumn("event_id", F.concat(F.lit("EX-"), F.lpad(F.col("event_sequence"), 5, "0")))
    .withColumn(
        "event_type",
        F.when((F.col("event_sequence") % 2) == 0, F.lit("view"))
         .otherwise(F.lit("click"))
    )
    .select("event_id", "event_type", "event_timestamp", "event_sequence")
)

print("Exercise 2 - Is streaming?:", df_ex_events.isStreaming)
df_ex_events.printSchema()

StatementMeta(, 795e056b-5a93-4d00-b892-9a3d6321b045, 21, Finished, Available, Finished, False)

Exercise 2 - Is streaming?: True
root
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = false)
 |-- event_timestamp: timestamp (nullable = true)
 |-- event_sequence: long (nullable = true)



In [ ]:
exercise_query_name = "day28_ex_events_memory"
exercise_checkpoint_path = "Files/day28_structured_streaming_basics/checkpoints/ex_events_memory"

for active_query in spark.streams.active:
    if active_query.name == exercise_query_name:
        active_query.stop()

# For lab reset only. Do not delete checkpoint blindly in production.
try:
    mssparkutils.fs.rm(exercise_checkpoint_path, True)
except Exception as e:
    print("Checkpoint cleanup skipped. Expected on first run.")

query_ex_events = (
    df_ex_events
    .writeStream
    .format("memory")
    .queryName(exercise_query_name)
    .outputMode("append")
    .option("checkpointLocation", exercise_checkpoint_path)
    .trigger(processingTime="5 seconds")
    .start()
)

time.sleep(12)
query_ex_events.stop()

spark.sql(f"SELECT * FROM {exercise_query_name} ORDER BY event_sequence LIMIT 20").show(truncate=False)

StatementMeta(, 795e056b-5a93-4d00-b892-9a3d6321b045, 23, Finished, Available, Finished, False)

+--------+----------+-----------------------+--------------+
|event_id|event_type|event_timestamp        |event_sequence|
+--------+----------+-----------------------+--------------+
|EX-00000|view      |2026-06-14 07:44:43.861|0             |
|EX-00001|click     |2026-06-14 07:44:44.361|1             |
|EX-00002|view      |2026-06-14 07:44:44.861|2             |
|EX-00003|click     |2026-06-14 07:44:45.361|3             |
|EX-00004|view      |2026-06-14 07:44:45.861|4             |
|EX-00005|click     |2026-06-14 07:44:46.361|5             |
|EX-00006|view      |2026-06-14 07:44:46.861|6             |
|EX-00007|click     |2026-06-14 07:44:47.361|7             |
|EX-00008|view      |2026-06-14 07:44:47.861|8             |
|EX-00009|click     |2026-06-14 07:44:48.361|9             |
|EX-00010|view      |2026-06-14 07:44:48.861|10            |
|EX-00011|click     |2026-06-14 07:44:49.361|11            |
|EX-00012|view      |2026-06-14 07:44:49.861|12            |
|EX-00013|click     |202

### Exercise 3: Aggregate streaming events by event type

สร้าง streaming aggregation เพื่อนับจำนวน event ต่อ `event_type`

Requirements:

- ใช้ `df_ex_events`
- `groupBy("event_type")`
- นับจำนวน records เป็น `event_count`
- เขียนเข้า memory sink ด้วย `outputMode("complete")`
- run สั้น ๆ แล้ว stop query

Expected result:

- ได้ result table ที่มี `event_type` และ `event_count`
- เข้าใจว่า aggregation result อาจ update ได้เรื่อย ๆ จึงใช้ `complete` mode ใน lab นี้

In [23]:
df_ex_event_counts = (
    df_ex_events
    .groupBy("event_type")
    .agg(F.count("*").alias("event_count"))
)

print("Exercise 3 - Is streaming?:", df_ex_event_counts.isStreaming)
df_ex_event_counts.printSchema()

StatementMeta(, 795e056b-5a93-4d00-b892-9a3d6321b045, 25, Finished, Available, Finished, False)

Exercise 3 - Is streaming?: True
root
 |-- event_type: string (nullable = false)
 |-- event_count: long (nullable = false)



In [ ]:
exercise_counts_query_name = "day28_ex_event_counts_memory"
exercise_counts_checkpoint_path = "Files/day28_structured_streaming_basics/checkpoints/ex_event_counts_memory"

for active_query in spark.streams.active:
    if active_query.name == exercise_counts_query_name:
        active_query.stop()

# For lab reset only. Do not delete checkpoint blindly in production.
try:
    mssparkutils.fs.rm(exercise_counts_checkpoint_path, True)
except Exception as e:
    print("Checkpoint cleanup skipped. Expected on first run.")

query_ex_counts = (
    df_ex_event_counts
    .writeStream
    .format("memory")
    .queryName(exercise_counts_query_name)
    .outputMode("complete")
    .option("checkpointLocation", exercise_counts_checkpoint_path)
    .trigger(processingTime="5 seconds")
    .start()
)

time.sleep(30)
query_ex_counts.stop()

spark.sql(f"SELECT * FROM {exercise_counts_query_name} ORDER BY event_type").show(truncate=False)

StatementMeta(, 795e056b-5a93-4d00-b892-9a3d6321b045, 27, Finished, Available, Finished, False)

+----------+-----------+
|event_type|event_count|
+----------+-----------+
|click     |12         |
|view      |12         |
+----------+-----------+



## Common mistakes

1. **คิดว่า `readStream` เริ่มอ่านข้อมูลทันที**

   `readStream` แค่สร้าง streaming DataFrame ไว้ก่อน ส่วน streaming query จะเริ่มทำงานจริงเมื่อเรียก `writeStream.start()`

2. **ลืม stop streaming query ใน notebook**

   ถ้าทดลองใน Fabric Notebook แล้วไม่ `stop()` query อาจมี active streaming query ค้างใน Spark session ทำให้ session ใช้ resource ต่อโดยไม่จำเป็น

3. **ใช้ memory sink เหมือน production sink**

   `memory` sink เหมาะกับ learning/debug เพราะ query ด้วย Spark SQL ได้ง่าย แต่ production ควรเขียนไป reliable sink เช่น Delta/Lakehouse table หรือระบบที่รองรับจริง

4. **ไม่กำหนด checkpointLocation**

   Checkpoint ช่วยให้ Spark จำ progress และ resume จากจุดเดิมได้เมื่อ streaming query เริ่มใหม่ ถ้าไม่มี checkpoint จะควบคุม restart behavior ได้ยาก

5. **ลบ checkpoint เพื่อแก้ error โดยไม่เข้าใจผลกระทบ**

   ใน lab เราอาจลบ checkpoint เพื่อ reset ตัวอย่างได้ แต่ใน production การลบ checkpoint อาจทำให้เกิด reprocess, duplicate หรือทำให้ state ของ aggregation หายได้

6. **เลือก output mode ไม่ตรงกับ transformation**

   Append mode เหมาะกับ append-only rows หรือผลลัพธ์ที่ไม่ต้องอัปเดตย้อนหลังแล้ว ส่วน streaming aggregation ที่ผลลัพธ์ยังเปลี่ยนแปลงได้ควรใช้ `update` หรือ `complete` และต้องตรวจว่า sink ที่ใช้ support output mode นั้นหรือไม่

7. **คิดว่า streaming คือ manual while loop**

   Structured Streaming ไม่ใช่การเขียน loop อ่าน source เอง แต่เป็น query ที่ Spark จัดการ micro-batch, progress และ execution lifecycle ให้


## Expected Output / Success Criteria

เมื่อจบ Day 28 ควรทำได้:

- อธิบายได้ว่า Structured Streaming คือ Spark streaming engine ที่ใช้ DataFrame/Spark SQL style
- แยกได้ว่า batch DataFrame กับ streaming DataFrame ต่างกันอย่างไร
- ใช้ `.isStreaming` และ `.printSchema()` ตรวจ streaming DataFrame ได้
- สร้าง streaming DataFrame จาก `rate` source สำหรับ lab ได้
- transform streaming DataFrame ด้วย `select`, `withColumn`, `filter` หรือ `groupBy` เบื้องต้นได้
- start/stop streaming query ด้วย `writeStream.start()` และ `query.stop()` ได้
- เข้าใจว่า `trigger`, `outputMode` และ `checkpointLocation` มีหน้าที่อะไร
- อ่าน output จาก memory sink เพื่อ validate ผลลัพธ์ใน notebook ได้
- อธิบายได้ว่า memory sink เป็น lab/debug sink ไม่ใช่ production sink
- พร้อมต่อยอด Day 29 เรื่อง streaming ingestion เข้า Bronze/Lakehouse พร้อม checkpoint และ reliability mindset

## Final takeaway

Structured Streaming ไม่ใช่แค่ syntax ใหม่ แต่คือ execution model ที่ทำให้ Spark process ข้อมูลใหม่แบบ incremental ได้

> Batch job มีจุดจบชัดเจน แต่ streaming query ต้องคิดเรื่อง source, sink, trigger, output mode, checkpoint และ lifecycle เสมอ

สำหรับ Day 28 สิ่งที่ต้องจำคือ:

- `readStream` สร้าง streaming DataFrame แต่ยังไม่เริ่ม query จริง
- `writeStream.start()` คือจุดที่ streaming query เริ่มทำงาน
- micro-batch คือวิธีที่ Spark process ข้อมูลใหม่เป็นรอบ ๆ
- checkpoint สำคัญต่อ progress tracking และ recovery
- output mode ต้องเลือกให้ตรงกับ transformation และ sink
- ใน notebook ต้อง stop query หลังทดลอง เพื่อไม่ให้ resource ค้าง

## Optional cleanup

In [ ]:
# Stop Day 28 active streaming queries if needed.
# for active_query in spark.streams.active:
#     if active_query.name and active_query.name.startswith("day28_"):
#         active_query.stop()

# Remove Day 28 checkpoint files if you want to reset the learning lab.
# Do not delete checkpoint blindly in production.
# try:
#     mssparkutils.fs.rm("Files/day28_structured_streaming_basics/checkpoints", True)
# except Exception as e:
#     print("Checkpoint cleanup skipped. Expected on first run.")